# Week 7 – "The Price is Right" Capstone Project

This week we fine-tune an open-source model to estimate product prices from descriptions.

## Order of play

- **Day 1:** QLoRA  
- **Day 2:** Prompt data and base model  
- **Day 3:** Train Part 1  
- **Day 4:** Train Part 2  
- **Day 5:** Eval

## Day 2: Prompt data and base model

Load the dataset from HuggingFace, set up the tokenizer, analyze token counts, and create prompts for fine-tuning.

In [ ]:
# Add week7 to path so we can import pricer
import sys
from pathlib import Path
for base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    week7_path = base / "week7"
    if (week7_path / "pricer").exists():
        sys.path.insert(0, str(week7_path))
        break

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.items import Item
from tqdm.notebook import tqdm
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

In [ ]:
LITE_MODE = False  # Set True for smaller dataset

load_dotenv(override=True)
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(hf_token, add_to_git_credential=True)
else:
    print("Warning: HF_TOKEN not set. Login manually with: login(token)")

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
items = train + val + test

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

In [ ]:
token_counts = [item.count_tokens(tokenizer) for item in tqdm(items)]

In [ ]:
plt.figure(figsize=(15, 6))
plt.title(f"Tokens in Summary: Avg {sum(token_counts)/len(token_counts):,.1f} and highest {max(token_counts):,}\n")
plt.xlabel("Number of tokens in summary")
plt.ylabel("Count")
plt.hist(token_counts, rwidth=0.7, color="skyblue", bins=range(0, 200, 10))
plt.show()

In [ ]:
CUTOFF = 110
cut = len([count for count in token_counts if count > CUTOFF])
print(f"With CUTOFF={CUTOFF}, we will truncate {cut:,} items which is {cut/len(items):.1%}")

In [ ]:
print("Sample summary:")
print(train[0].summary)

In [ ]:
for item in tqdm(train + val):
    item.make_prompts(tokenizer, CUTOFF, True)
for item in tqdm(test):
    item.make_prompts(tokenizer, CUTOFF, False)

In [ ]:
print("PROMPT:")
print(test[0].prompt)
print("\nCOMPLETION:")
print(test[0].completion)

In [ ]:
prompt_token_counts = [item.count_prompt_tokens(tokenizer) for item in tqdm(items)]

In [ ]:
plt.figure(figsize=(15, 6))
plt.title(f"Prompt+Completion Tokens: Avg {sum(prompt_token_counts)/len(prompt_token_counts):,.1f} and highest {max(prompt_token_counts):,}\n")
plt.xlabel("Number of tokens (prompt + completion)")
plt.ylabel("Count")
plt.hist(prompt_token_counts, rwidth=0.7, color="gold", bins=range(0, 200, 10))
plt.show()

In [ ]:
# Optional: push prompts to your HuggingFace account for training
# username = "your-hf-username"
# dataset = f"{username}/items_prompts_lite" if LITE_MODE else f"{username}/items_prompts_full"
# Item.push_prompts_to_hub(dataset, train, val, test)

# Or use Ed's public datasets:
# https://huggingface.co/datasets/ed-donner/items_prompts_lite
# https://huggingface.co/datasets/ed-donner/items_prompts_full

## Days 3 & 4: Training with QLoRA

Fine-tune Llama 3.2 3B with QLoRA (4-bit quantization + LoRA). **Requires GPU** (e.g. Google Colab T4/A100).

Training notebooks:
- [Day 3 & 4 Colab](https://colab.research.google.com/drive/1fBTm_jzrFGr88PDOFTQF7JIlQ1JW15BG)
- [Day 5 Eval Colab](https://colab.research.google.com/drive/16e8aY_BlHjzzcR-2dCyDMCPdOQ8XeN1e)

Below: load prompts from HuggingFace and run training locally (if you have a GPU).

In [ ]:
# Training setup - run in Colab or on a GPU machine
# Uncomment and run to load prompts dataset and train

# from datasets import load_dataset
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
# from trl import SFTTrainer, SFTConfig
# import torch

# PROMPTS_DATASET = "ed-donner/items_prompts_lite"  # or items_prompts_full
# dataset = load_dataset(PROMPTS_DATASET)
# train_data = dataset["train"]
# val_data = dataset["val"]

# def format_example(ex): return {"text": ex["prompt"] + ex["completion"]}
# train_formatted = train_data.map(format_example)
# val_formatted = val_data.map(format_example)

# # Load base model with 4-bit quantization
# bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
# model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")
# tokenizer.pad_token = tokenizer.eos_token

# # LoRA config
# model = prepare_model_for_kbit_training(model)
# lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj","v_proj"], lora_dropout=0.05, task_type="CAUSAL_LM")
# model = get_peft_model(model, lora_config)

# # SFTConfig + SFTTrainer - then trainer.train()
print("See Colab links above for full training pipeline.")

## Day 5: Evaluation

Evaluate the fine-tuned model: load it, run predictions on test data, compute mean absolute error.

In [ ]:
# Evaluation with week7 pricer.evaluator (requires fine-tuned model)
# Run from week7 directory or ensure week7 is on path

try:
    from pricer.evaluator import evaluate, Tester
    def fine_tuned_pricer(item):
        # Replace with your actual model inference logic
        # e.g. load model, tokenizer, run generate on item.test_prompt()
        return item.price  # placeholder
    fine_tuned_pricer.__name__ = "fine_tuned_llama"
    # evaluate(fine_tuned_pricer, test, size=200)
    print("evaluate() and Tester available. Define your pricer and run evaluate(pricer, test).")
except ImportError as e:
    print(f"pricer.evaluator not found: {e}. Run from repo root or add week7 to path.")

## Results: Model comparison

Prediction error (mean absolute error) across baselines, traditional ML, LLMs, and fine-tuned models.

In [ ]:
import plotly.graph_objects as go

results = [
    ("Constant", "gray", 106.18),
    ("Linear Regression", "gray", 101.56),
    ("NLP + LR", "gray", 76.81),
    ("Random Forest", "gray", 72.28),
    ("XGBoost", "gray", 68.23),
    ("Human (Ed)", "black", 87.62),
    ("Neural Network", "orange", 63.97),
    ("GPT 4.1 Nano", "slateblue", 62.51),
    ("Grok 4.1 Fast", "slateblue", 57.62),
    ("Gemini 3 Pro", "slateblue", 50.54),
    ("Claude 4.5 Sonnet", "slateblue", 47.10),
    ("GPT 5.1", "slateblue", 44.74),
    ("GPT 4.1 Nano (Fine-tuned)", "skyblue", 75.91),
    ("Deep Neural Network", "orange", 46.49),
    ("Base Llama 3.2 4 bit", "darkred", 110.72),
    ("Fine-tuned Lite", "red", 65.40),
    ("Fine-tuned Full", "red", 39.85)
]

labels, colors, values = zip(*results)

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))

fig.update_layout(
    title="Prediction error from each model",
    yaxis=dict(range=[0, max(values)], title="Error"),
    xaxis=dict(tickangle=-45),
    width=1000,
    height=800
)

fig.show()